# Time Series Analysis — Gamma at NTM Strikes

Análisis de estacionariedad y primeras diferencias del gamma interpolado en cuantiles de moneyness, para calls y puts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.widgets import Button
import math
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

PARQUET_PATH = r"C:\Users\pablo.esparcia\Documents\OptionMetrics\output\time_series\quantile_gamma.parquet"

## 1. Carga de datos

In [ ]:
quantile_gammas = pd.read_parquet(PARQUET_PATH).set_index("Date")
quantile_gammas.index = pd.to_datetime(quantile_gammas.index)

print(f"Periodo: {quantile_gammas.index.min().date()} → {quantile_gammas.index.max().date()}")
print(f"Observaciones: {len(quantile_gammas):,} | Series: {quantile_gammas.shape[1]}")
quantile_gammas.describe().round(4)

## 2. Series en niveles

In [ ]:
cols = list(quantile_gammas.columns)
idx  = [0]

fig, ax = plt.subplots(figsize=(14, 4))
plt.subplots_adjust(bottom=0.25)

def update_levels():
    col = cols[idx[0]]
    ax.cla()
    ax.plot(quantile_gammas.index, quantile_gammas[col], linewidth=0.8)
    ax.set_title(f"{col}  ({idx[0]+1}/{len(cols)}) — nivel")
    ax.set_ylabel("gamma")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', rotation=45)
    fig.canvas.draw()

ax_prev = plt.axes([0.35, 0.03, 0.1, 0.05])
ax_next = plt.axes([0.55, 0.03, 0.1, 0.05])
Button(ax_prev, '◀ Anterior').on_clicked(lambda _: [idx.__setitem__(0, (idx[0]-1) % len(cols)), update_levels()])
Button(ax_next, 'Siguiente ▶').on_clicked(lambda _: [idx.__setitem__(0, (idx[0]+1) % len(cols)), update_levels()])

update_levels()
plt.show()

## 3. Test ADF — Niveles I(0)

Contraste de raíz unitaria con constante, tendencia lineal y término cuadrático (`regression='ctt'`).

- **H₀**: la serie tiene raíz unitaria (no estacionaria)
- Rechazamos H₀ si p-value < 5 %

In [ ]:
rows = []
for col in quantile_gammas.columns:
    res = adfuller(quantile_gammas[col].dropna(), regression="ctt")
    rows.append({"serie": col, "p-value (%)": round(res[1]*100, 4), "estacionaria": res[1] < 0.05})


adf_diff = pd.DataFrame(rows).set_index("serie")
adf_diff.style.map(lambda v: "background-color: #d4edda; color: #155724" if v else "background-color: #f8d7da; color: #721c24",
                         subset=["estacionaria"])

## 4. Primeras diferencias porcentuales

Calculamos `pct_change()` para obtener las series I(1) y comprobamos si son estacionarias.

In [ ]:
quantile_diff = quantile_gammas.pct_change().iloc[1:-1]
quantile_diff.describe().round(4)

## 5. Test ADF — Primeras diferencias I(1)

In [ ]:
rows = []
for col in quantile_diff.columns:
    res = adfuller(quantile_diff[col].dropna(), regression="ctt")
    rows.append({"serie": col, "p-value (%)": round(res[1]*100, 4), "estacionaria": res[1] < 0.05})

adf_diff = pd.DataFrame(rows).set_index("serie")
adf_diff.style.map(lambda v: "background-color: #d4edda; color: #155724" if v else "background-color: #f8d7da; color: #721c24",
                         subset=["estacionaria"])

## 6. Series en primeras diferencias

In [ ]:
cols_d = list(quantile_diff.columns)
idx_d  = [0]

fig, ax = plt.subplots(figsize=(14, 4))
plt.subplots_adjust(bottom=0.25)

def update_diff():
    col = cols_d[idx_d[0]]
    ax.cla()
    ax.plot(quantile_diff.index, quantile_diff[col], linewidth=0.8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(f"{col}  ({idx_d[0]+1}/{len(cols_d)}) — I(1)")
    ax.set_ylabel("Δ gamma (%)")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', rotation=45)
    fig.canvas.draw()

ax_prev = plt.axes([0.35, 0.03, 0.1, 0.05])
ax_next = plt.axes([0.55, 0.03, 0.1, 0.05])
Button(ax_prev, '◀ Anterior').on_clicked(lambda _: [idx_d.__setitem__(0, (idx_d[0]-1) % len(cols_d)), update_diff()])
Button(ax_next, 'Siguiente ▶').on_clicked(lambda _: [idx_d.__setitem__(0, (idx_d[0]+1) % len(cols_d)), update_diff()])

update_diff()
plt.show()

## 7. Análisis de autocorrelaciones. 
#### 7.1. Gráficas de ACF I(1):

In [ ]:
n = len(cols_d)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3), constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(cols_d):
    plot_acf(quantile_diff[col].dropna(), lags=30, ax=axes[i], zero=False, alpha=0.05)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("ACF — Primeras diferencias I(0)", fontsize=12, fontweight="bold")
plt.show()


#### 7.2. Gráficas de PACF I(1):

In [ ]:
n = len(cols_d)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3), constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(cols_d):
    plot_pacf(quantile_diff[col], lags=30, ax=axes[i], zero=False, method="ols", alpha=0.05)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("PACF — Primeras diferencias I(1)", fontsize=12, fontweight="bold")
plt.show()


#### 7.3. Gráficas de ACF I(1) con diferencias al cuadrado:

In [ ]:
n = len(cols_d)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3), constrained_layout=True)
axes = axes.flatten()

quantile_diff_sqrt = quantile_diff**2

for i, col in enumerate(cols_d):
    plot_acf(quantile_diff_sqrt[col].dropna(), lags=30, ax=axes[i], zero=False, alpha=0.05)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("ACF — Primeras diferencias^2 I(0)", fontsize=12, fontweight="bold")
plt.show()


#### 7.4. Gráficas de PACF I(1) con diferencias al cuadrado:

In [ ]:
n = len(cols_d)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3), constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(cols_d):
    plot_pacf(quantile_diff_sqrt[col], lags=30, ax=axes[i], zero=False, method="ols", alpha=0.05)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("PACF — Primeras diferencias^2 I(1)", fontsize=12, fontweight="bold")
plt.show()
